In [19]:
# ============================================
# Survival Analysis Pipeline for Plasma Proteomics
# Cox regression + Kaplan-Meier curves
# ============================================

import os
import re
import glob
import numpy as np
import pandas as pd

from lifelines import CoxPHFitter
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

In [ ]:
# ============================================
# 1. LOAD INPUT TABLES
# ============================================

sample_info = pd.read_csv("input/sample_info.tsv", sep="\t")
survival_info = pd.read_csv("input/survival_info.tsv", sep="\t")
meta = pd.read_csv("input/meta.tsv", sep="\t")
quant = pd.read_csv("input/plasma_prot_quant.tsv", sep="\t")

/var/folders/7h/dvymb3m50bzg0j9yyrdvvn000000gn/T/ipykernel_11801/4240999496.py:8: DtypeWarning: Columns (0: [254] XCHEJF1-254_20250824_plasma_S254_250ng_41p5min_DIA_r2.raw.PG.MS1Quantity, 1: [254] XCHEJF1-254_20250824_plasma_S254_250ng_41p5min_DIA_r2.raw.PG.MS2Quantity) have mixed types. Specify dtype option on import or set low_memory=False.
  quant = pd.read_csv("input/plasma_prot_quant.tsv", sep="\t")


In [6]:
# ============================================
# 2. CLEAN PROTEIN NAMES
# ============================================

quant["PG.Genes"] = quant["PG.Genes"].apply(
    lambda x: f"{x.split(';')[0]} genes"
    if isinstance(x, str) and ";" in x
    else x
)

# Remove duplicated protein names
quant = quant.drop_duplicates(subset="PG.Genes")



In [ ]:
# ============================================
# 3. MAP QUANT COLUMNS -> CF IDs
# ============================================

mapping = dict(zip(meta["R.FileName"], meta["Sample ID"]))

quant_cols = quant.columns[1:]

col_to_cf = {}

for col in quant_cols:
    for rfile, cfid in mapping.items():
        if rfile in col:
            col_to_cf[col] = cfid
            break

# Keep mapped columns
quant_filtered = quant[["PG.Genes"] + list(col_to_cf.keys())]

# Rename quant columns -> CF IDs
quant_filtered = quant_filtered.rename(columns=col_to_cf)
quant_filtered.head()

,PG.Genes,XCHEJF_001,XCHEJF_002,XCHEJF_003,XCHEJF_004,XCHEJF_005,XCHEJF_006,XCHEJF_007,XCHEJF_008,XCHEJF_009,...,XCHEJF_059,XCHEJF_098,XCHEJF_174,XCHEJF_233,XCHEJF_236,XCHEJF_241,XCHEJF_243,XCHEJF_244,XCHEJF_245,XCHEJF_254
0,ARF5,1,0,0,1,0,1,0,0,0,...,10 326 348 114 013 600,1 163 571 548 461 910,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FUCA2,8,8,7,10,6,6,8,7,8,...,1 933 536 865 234 370,19 482 252 197 265 600,NaN,15 846 005 859 375,5 821 880 493 164 060,8 247 229 614 257 810,1 908 680 419 921 870,8 914 685 668 945 310,13 067 958 984 375,NaN
2,SEMA3F,2,2,2,1,1,1,1,2,1,...,2 962 778 625 488 280,2 633 799 133 300 780,NaN,3 033 146 667 480 460,2 950 328 063 964 840,18 923 497 009 277 300,28 078 668 212 890 600,4 567 154 235 839 840,35 508 428 955 078 100,NaN
3,CEACAM7,0,0,0,0,0,0,0,0,0,...,5 813 136 291 503 900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 879 190 063 476 500,NaN
4,KRT33A,0,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,5 000 421 447 753 900,NaN,NaN,NaN,NaN


In [ ]:
# ============================================
# 4. BUILD EXPRESSION MATRIX
# ============================================

expr = quant_filtered.set_index("PG.Genes").T

# Remove duplicated samples
expr = expr[~expr.index.duplicated(keep="first")]

# Numeric
expr = expr.apply(pd.to_numeric, errors="coerce")

# Log2 transform
expr = np.log2(expr + 1)

# Remove duplicated proteins
expr = expr.loc[:, ~expr.columns.duplicated()]

print("Expression matrix shape:", expr.shape)

Expression matrix shape: (254, 2000)


In [9]:
# ============================================
# 5. PREPARE SAMPLE INFO
# ============================================

# Keep baseline only (recommended)
sample_info = sample_info[sample_info["TP"] == "T0"].copy()

# Merge survival + sample info
clinical = sample_info.merge(
    survival_info,
    left_on="Sample ID",
    right_on="patient_code",
    how="inner"
)

In [ ]:
# ============================================
# 6. ALIGN EXPRESSION + CLINICAL
# ============================================

common = expr.index.intersection(clinical["CF ID"])

expr = expr.loc[common]

clinical = clinical.set_index("CF ID")
clinical = clinical.loc[common]

print("Samples for survival analysis:", len(common))

Samples for survival analysis: 122


In [27]:
# ============================================
# 7. CREATE PFS EVENT COLUMN
# ============================================

# PFS:
# 0 = censored
# 1 or 2 = event

clinical["pfs_event"] = clinical["dfs"].apply(
    lambda x: 0 if x == 0 else 1
)

clinical.head()

,96-well,No.,Sample ID,TP,Sex,Pep. Conc. ug/uL,250ng uL,750ng uL,patient_code,dfs,tdfs,surv,tsurv,Unnamed: 5,Unnamed: 6,pfs_event
XCHEJF_001,1_A1,18,SWE001,T0,M,0.1750,1.43,4.29,SWE001,0,9.97,0,9.97,NaN,NaN,0
XCHEJF_003,1_C1,20,SWE002,T0,M,0.2164,1.16,3.47,SWE002,1,6.05,0,6.05,NaN,NaN,1
XCHEJF_005,1_E1,22,SWE004,T0,M,0.2236,1.12,3.35,SWE004,1,9.77,1,16.68,NaN,NaN,1
XCHEJF_007,1_G1,24,SWE005,T0,M,0.2154,1.16,3.48,SWE005,0,10.20,0,10.20,NaN,NaN,0
XCHEJF_009,1_A2,26,SWE006,T0,F,0.2316,1.08,3.24,SWE006,0,21.41,0,21.41,NaN,NaN,0


_helper functions_

In [39]:
# ============================================
# OUTPUT DIRS
# ============================================

os.makedirs("output/cox_regression", exist_ok=True)
os.makedirs("output/km_curves", exist_ok=True)

# ============================================
# FUNCTION: COX REGRESSION
# ============================================

def run_cox_regression(
    expr,
    clinical,
    genes,
    time_col,
    event_col
):

    results = []

    for gene in genes:

        if gene not in expr.columns:
            continue

        df = pd.DataFrame({
            "time": clinical[time_col],
            "event": clinical[event_col],
            "expression": expr[gene]
        })

        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna()

        if df.shape[0] < 20:
            continue

        try:

            cph = CoxPHFitter()

            cph.fit(
                df,
                duration_col="time",
                event_col="event"
            )

            s = cph.summary.iloc[0]

            results.append({
                "Protein": gene,
                "coef": s["coef"],
                "HR": s["exp(coef)"],
                "pvalue": s["p"],
                "CI_lower": s["exp(coef) lower 95%"],
                "CI_upper": s["exp(coef) upper 95%"]
            })

        except Exception as e:
            print(f"Failed Cox: {gene} | {e}")

    res_df = pd.DataFrame(results)

    if len(res_df) == 0:
        return res_df

    res_df["adj_pvalue"] = multipletests(
        res_df["pvalue"],
        method="fdr_bh"
    )[1]

    res_df = res_df.sort_values("adj_pvalue")

    return res_df

# ============================================
# FUNCTION: KM CURVES MULTIPAGE PDF
# ============================================

def save_km_curves(
    expr,
    clinical,
    genes,
    time_col,
    event_col,
    output_pdf
):

    kmf = KaplanMeierFitter()

    with PdfPages(output_pdf) as pdf:

        for gene in genes:

            if gene not in expr.columns:
                continue

            df = pd.DataFrame({
                "Sample ID": clinical["Sample ID"],
                "time": clinical[time_col],
                "event": clinical[event_col],
                "expression": expr[gene]
            })

            # -----------------------------------
            # REMOVE DUPLICATED PATIENTS
            # -----------------------------------
            df = df.drop_duplicates(
                subset="Sample ID",
                keep="first"
            )

            # -----------------------------------
            # FORCE NUMERIC
            # -----------------------------------
            df["time"] = pd.to_numeric(
                df["time"],
                errors="coerce"
            )
            df["event"] = pd.to_numeric(
                df["event"],
                errors="coerce"
            )

            df["expression"] = pd.to_numeric(
                df["expression"],
                errors="coerce"
            )

            # -----------------------------------
            # CLEAN BAD VALUES
            # -----------------------------------

            df = df.replace([np.inf, -np.inf], np.nan)
            df = df.dropna(subset=[
                "time",
                "event",
                "expression"
            ])

            df["event"] = df["event"].astype(int)

            # -----------------------------------
            # SKIP LOW QUALITY DATA
            # -----------------------------------

            if df.shape[0] < 20:
                continue

            if df["expression"].nunique() < 2:
                continue

            # -----------------------------------
            # MEAN SPLIT
            # -----------------------------------

            cutoff = df["expression"].mean()

            df["group"] = np.where(
                df["expression"] >= cutoff,
                "High",
                "Low"
            )

            high = df[df["group"] == "High"]
            low = df[df["group"] == "Low"]

            # -----------------------------------
            # Logrank
            # -----------------------------------

            try:

                lr = logrank_test(
                    high["time"],
                    low["time"],
                    event_observed_A=high["event"],
                    event_observed_B=low["event"]
                )

                pval = lr.p_value

                # -----------------------------------
                # Plot
                # -----------------------------------

                plt.figure(figsize=(5, 5))

                for group, color in zip(
                    ["High", "Low"],
                    ["#fc9272", "#9ecae1"]
                ):

                    sub = df[df["group"] == group]

                    kmf.fit(
                        sub["time"],
                        event_observed=sub["event"],
                        label=f"{group}"
                    )

                    kmf.plot_survival_function(
                        ci_show=False,
                        linewidth=2,
                        color=color
                    )

                plt.title(
                    f"{gene}\nLog-rank p = {pval:.2e}",
                    fontsize=11,
                    color="black"
                )

                plt.xlabel("Time", color="black")
                plt.ylabel("Survival probability", color="black")

                plt.xticks(color="black")
                plt.yticks(color="black")

                plt.legend(frameon=False)

                sns.despine()

                plt.tight_layout()

                pdf.savefig()
                plt.close()

            except Exception as e:
                print(f"Failed KM: {gene} | {e}")



_Running through the significant proteins_

In [40]:
# ============================================
# LOOP THROUGH LIMMA OUTPUTS
# ============================================

limma_files = glob.glob("output/limma/limma_diff_prot-*.tsv")

print(f"Found {len(limma_files)} limma outputs")

for file in limma_files:

    print("\n==============================")
    print("Processing:", file)

    limma = pd.read_csv(file, sep="\t")

    # ----------------------------------------
    # Select significant proteins
    # ----------------------------------------

    sig = limma[
        (limma["adj_pvalue"] < 0.05) &
        (limma["log2FC"].abs() > 0.5)
    ].copy()

    genes = sig["Protein"].dropna().unique().tolist()

    print(f"Significant proteins: {len(genes)}")

    if len(genes) == 0:
        continue

    # ----------------------------------------
    # Extract filename label
    # ----------------------------------------

    base = os.path.basename(file)

    label = (
        base
        .replace("limma_diff_prot-", "")
        .replace(".tsv", "")
    )

    # ========================================
    # COX REGRESSION
    # ========================================

    cox_res = run_cox_regression(
        expr=expr,
        clinical=clinical,
        genes=genes,
        time_col="tsurv",
        event_col="surv"
    )

    cox_out = f"output/cox_regression/cox-{label}.tsv"

    cox_res.to_csv(
        cox_out,
        sep="\t",
        index=False
    )

    print("Saved Cox:", cox_out)

    # ========================================
    # KM CURVES
    # ========================================

    km_pdf = f"output/km_curves/km-{label}.pdf"

    save_km_curves(
        expr=expr,
        clinical=clinical,
        genes=genes,
        time_col="tsurv",
        event_col="surv",
        output_pdf=km_pdf
    )

    print("Saved KM:", km_pdf)

print("\nDONE")

Found 14 limma outputs

Processing: output/limma/limma_diff_prot-T1_vs_T0_all.tsv
Significant proteins: 2
Saved Cox: output/cox_regression/cox-T1_vs_T0_all.tsv
Saved KM: output/km_curves/km-T1_vs_T0_all.pdf

Processing: output/limma/limma_diff_prot-T1_vs_T0_tox.tsv
Significant proteins: 2
Saved Cox: output/cox_regression/cox-T1_vs_T0_tox.tsv


/Users/szabolcshetey/Desktop/Hanna/GDefiner/scripts/plasma_proteomics/.venv/lib/python3.13/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column expression have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['event'].astype(bool)
>>> print(df.loc[events, 'expression'].var())
>>> print(df.loc[~events, 'expression'].var())

A very low variance means that the column expression completely determines whether a subject dies or not. See https://stats.stackexchange.com/questions/11109/how-to-deal-with-perfect-separation-in-logistic-regression.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/Users/szabolcshetey/Desktop/Hanna/GDefiner/scripts/plasma_proteomics/.venv/lib/python3.13/site-packages/lifelines/fitters/coxph_fitter.py:1607: ConvergenceWarning: Newton-Raphson convergence completed successfully but norm(delta) 

Saved KM: output/km_curves/km-T1_vs_T0_tox.pdf

Processing: output/limma/limma_diff_prot-T2_vs_T0_F.tsv
Significant proteins: 19
Failed Cox: PIBF1 | delta contains nan value(s). Convergence halted. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-model
Saved Cox: output/cox_regression/cox-T2_vs_T0_F.tsv


/Users/szabolcshetey/Desktop/Hanna/GDefiner/scripts/plasma_proteomics/.venv/lib/python3.13/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['expression'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/Users/szabolcshetey/Desktop/Hanna/GDefiner/scripts/plasma_proteomics/.venv/lib/python3.13/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column expression have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['event'].astype(bool)
>>> print(df.loc[events, 'expression'].var())
>>> print(df.loc[~events, 'expression'].var())

A very low variance means that the column expression completely determines

Saved KM: output/km_curves/km-T2_vs_T0_F.pdf

Processing: output/limma/limma_diff_prot-T2_vs_T0_all.tsv
Significant proteins: 1
Saved Cox: output/cox_regression/cox-T2_vs_T0_all.tsv
Saved KM: output/km_curves/km-T2_vs_T0_all.pdf

Processing: output/limma/limma_diff_prot-T1_vs_T0_notox.tsv
Significant proteins: 3
Failed Cox: LRIG3 | delta contains nan value(s). Convergence halted. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-model
Saved Cox: output/cox_regression/cox-T1_vs_T0_notox.tsv


...

Saved KM: output/km_curves/km-T1_vs_T0_M.pdf

Processing: output/limma/limma_diff_prot-M_vs_F_all.tsv
Significant proteins: 2
Saved Cox: output/cox_regression/cox-M_vs_F_all.tsv
Saved KM: output/km_curves/km-M_vs_F_all.pdf

Processing: output/limma/limma_diff_prot-T2_vs_T0_M.tsv
Significant proteins: 3
Saved Cox: output/cox_regression/cox-T2_vs_T0_M.tsv


/Users/szabolcshetey/Desktop/Hanna/GDefiner/scripts/plasma_proteomics/.venv/lib/python3.13/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column expression have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['event'].astype(bool)
>>> print(df.loc[events, 'expression'].var())
>>> print(df.loc[~events, 'expression'].var())

A very low variance means that the column expression completely determines whether a subject dies or not. See https://stats.stackexchange.com/questions/11109/how-to-deal-with-perfect-separation-in-logistic-regression.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/Users/szabolcshetey/Desktop/Hanna/GDefiner/scripts/plasma_proteomics/.venv/lib/python3.13/site-packages/lifelines/fitters/coxph_fitter.py:1607: ConvergenceWarning: Newton-Raphson convergence completed successfully but norm(delta) 

Saved KM: output/km_curves/km-T2_vs_T0_M.pdf

Processing: output/limma/limma_diff_prot-M_vs_F_T0_notox.tsv
Significant proteins: 0

Processing: output/limma/limma_diff_prot-tox_vs_notox_T0.tsv
Significant proteins: 0

Processing: output/limma/limma_diff_prot-T2_vs_T0_F_tox.tsv
Significant proteins: 10
Failed Cox: PIBF1 | delta contains nan value(s). Convergence halted. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-model
Saved Cox: output/cox_regression/cox-T2_vs_T0_F_tox.tsv


/Users/szabolcshetey/Desktop/Hanna/GDefiner/scripts/plasma_proteomics/.venv/lib/python3.13/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column expression have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['event'].astype(bool)
>>> print(df.loc[events, 'expression'].var())
>>> print(df.loc[~events, 'expression'].var())

A very low variance means that the column expression completely determines whether a subject dies or not. See https://stats.stackexchange.com/questions/11109/how-to-deal-with-perfect-separation-in-logistic-regression.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/Users/szabolcshetey/Desktop/Hanna/GDefiner/scripts/plasma_proteomics/.venv/lib/python3.13/site-packages/lifelines/fitters/coxph_fitter.py:1607: ConvergenceWarning: Newton-Raphson convergence completed successfully but norm(delta) 

Saved KM: output/km_curves/km-T2_vs_T0_F_tox.pdf

DONE


In [42]:


# ============================================
# OUTPUT DIRECTORY
# ============================================

os.makedirs("output/cox_regression", exist_ok=True)

# ============================================
# LOAD COX FILES
# ============================================

cox_files = glob.glob("output/cox_regression/cox-*.tsv")

print(f"Found {len(cox_files)} Cox result files")

# ============================================
# LOOP THROUGH FILES
# ============================================

for file in cox_files:

    print(f"\nProcessing: {file}")

    try:

        # ------------------------------------
        # LOAD
        # ------------------------------------

        df = pd.read_csv(file, sep="\t")

        if df.shape[0] == 0:
            print("Empty file")
            continue

        # ------------------------------------
        # CLEAN
        # ------------------------------------

        df = df.replace([np.inf, -np.inf], np.nan)

        df = df.dropna(subset=[
            "Protein",
            "HR",
            "adj_pvalue"
        ])

        if df.shape[0] == 0:
            continue

        # ------------------------------------
        # SORT
        # ------------------------------------

        df = df.sort_values(
            "adj_pvalue",
            ascending=True
        )

        # Top proteins
        top_n = min(20, df.shape[0])

        df = df.head(top_n).copy()

        # ------------------------------------
        # SIGNIFICANCE LABELS
        # ------------------------------------

        def significance_label(p):

            if p < 0.001:
                return "***"
            elif p < 0.01:
                return "**"
            elif p < 0.05:
                return "*"
            else:
                return "n.s."

        df["sig_label"] = df["adj_pvalue"].apply(
            significance_label
        )

        # ------------------------------------
        # COLORS
        # ------------------------------------

        df["direction"] = np.where(
            df["HR"] > 1,
            "Risk",
            "Protective"
        )

        palette = {
            "Risk": "#de2d26",
            "Protective": "#31a354"
        }

        # ------------------------------------
        # PLOT
        # ------------------------------------

        plt.figure(figsize=(8, max(5, len(df) * 0.4)))

        ax = sns.barplot(
            data=df,
            y="Protein",
            x="HR",
            hue="direction",
            dodge=False,
            palette=palette,
            edgecolor="black"
        )

        # ------------------------------------
        # ADD SIGNIFICANCE LABELS
        # ------------------------------------

        for i, row in df.iterrows():

            ax.text(
                row["HR"] + 0.03,
                list(df.index).index(i),
                row["sig_label"],
                va="center",
                fontsize=10,
                color="black"
            )

        # ------------------------------------
        # STYLING
        # ------------------------------------

        plt.axvline(
            1,
            linestyle="--",
            color="black",
            linewidth=1
        )

        title = os.path.basename(file)
        title = title.replace("cox-", "")
        title = title.replace(".tsv", "")

        plt.title(
            title,
            fontsize=13,
            color="black"
        )

        plt.xlabel(
            "Hazard Ratio (HR)",
            color="black"
        )

        plt.ylabel(
            "",
            color="black"
        )

        plt.xticks(color="black")
        plt.yticks(color="black")

        plt.legend(
            title="",
            frameon=False,
            bbox_to_anchor=(1.02, 1),
            loc="upper left"
        )

        sns.despine()

        plt.tight_layout()

        # ------------------------------------
        # SAVE
        # ------------------------------------

        out_pdf = os.path.join(
            "output/cox_regression",
            title + ".pdf"
        )

        plt.savefig(out_pdf)

        plt.close()

        print("Saved:", out_pdf)

    except Exception as e:

        print(f"FAILED: {file}")
        print(e)

print("\nDONE")

Found 10 Cox result files

Processing: output/cox_regression/cox-T2_vs_T0_F_tox.tsv
Saved: output/cox_regression/T2_vs_T0_F_tox.pdf

Processing: output/cox_regression/cox-M_vs_F_T0_tox.tsv
Saved: output/cox_regression/M_vs_F_T0_tox.pdf

Processing: output/cox_regression/cox-T2_vs_T0_F.tsv

Processing: output/cox_regression/cox-T2_vs_T0_all.tsv
Saved: output/cox_regression/T2_vs_T0_all.pdf

Processing: output/cox_regression/cox-T1_vs_T0_notox.tsv
Saved: output/cox_regression/T1_vs_T0_notox.pdf

Processing: output/cox_regression/cox-T1_vs_T0_tox.tsv
Saved: output/cox_regression/T1_vs_T0_tox.pdf

Processing: output/cox_regression/cox-T1_vs_T0_all.tsv
Saved: output/cox_regression/T1_vs_T0_all.pdf

Processing: output/cox_regression/cox-T2_vs_T0_M.tsv
Saved: output/cox_regression/T2_vs_T0_M.pdf

Processing: output/cox_regression/cox-M_vs_F_all.tsv
Saved: output/cox_regression/M_vs_F_all.pdf

Processing: output/cox_regression/cox-T1_vs_T0_M.tsv
Saved: output/cox_regression/T1_vs_T0_M.pdf

DON